**What's in this notebook?** This notebook reproduces flux vacuum dataset B from [arXiv:2501.03984](https://arxiv.org/abs/2501.03984) (*Deep observations of the Type IIB flux landscape*) using JAXVacua.

- **Model setup** — load the CP⁴[1,1,1,6,9][18] model at LCS with the correct `a_matrix`, verified against the paper.
- **Dataset B** — load and inspect the 12,196 stored SUSY vacua (N_flux ∈ [2,10], Im(τ) ∈ [0.87, 10], Im(zⁱ) ∈ [2, 5]).
- **Bounding box analysis** — compute eigenvalue-based bounding boxes; explain why the full scan requires the h₂-outer streaming algorithm.
- **Test scan** — enumerate all flux vacua with N_flux ≤ 4, s ≥ 2 using `use_linearised_shifts=True`; expect 42 solutions (verified against Dataset B).
- **Direct verification** — Newton-refine 500 stored Dataset B solutions to confirm model correctness.
- **Cross-check** — verify that every flux vector found by the test scan is contained in Dataset B.
- **Full Dataset B scan** — complete scan setup (gated by `RUN_FULL_SCAN = True`).
- **Summary comparison** — VEV distribution plots and final recall/precision summary.

(*Created:* Andreas Schachner, March 2025)

In [ ]:
import warnings
import numpy as np

import jax
import jax.numpy as jnp
from jax import jit
from jax import Array
from jax.typing import ArrayLike

import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)

import jaxvacua as jvc

warnings.filterwarnings('ignore')

In [ ]:
h12 = 2
model = jvc.FluxVacuaFinder(h12=h12, model_ID=1)
model

In [ ]:
model.lcs_tree.a_matrix = jnp.array([[4.5,1.5],[1.5,0.]])

## Load Dataset B

Load the stored solutions from `dataset_B.p` (12,196 SUSY vacua).  
Columns: `[Re(z1), Im(z1), Re(z2), Im(z2), Re(τ), Im(τ), f1..f6, h1..h6]`.

In [ ]:
datsB = jvc.util.load_zipped_pickle("./dataset_B.p")

# VEVs
z1  = datsB[:, 0] + 1j * datsB[:, 1]
z2  = datsB[:, 2] + 1j * datsB[:, 3]
tau_B = datsB[:, 4] + 1j * datsB[:, 5]

# Flux vectors
f_B = datsB[:, 6:12]    # RR fluxes (6 components)
h_B = datsB[:, 12:18]   # NSNS fluxes (6 components)
flux_B = datsB[:, 6:18]  # full [f|h] (12 components)

# Tadpole: N_flux = |f^T Σ h|, Σ = [[0, I3], [-I3, 0]]
sigma_np = np.array([[0,0,0,1,0,0],[0,0,0,0,1,0],[0,0,0,0,0,1],
                     [-1,0,0,0,0,0],[0,-1,0,0,0,0],[0,0,-1,0,0,0]], float)
Nflux_B = np.abs(np.einsum('ki,ij,kj->k', f_B, sigma_np, h_B))

print(f"Dataset B: {len(datsB)} solutions")
print(f"N_flux range:   {int(Nflux_B.min())} – {int(Nflux_B.max())}")
print(f"Im(τ)  range:   {np.imag(tau_B).min():.3f} – {np.imag(tau_B).max():.3f}")
print(f"Im(z1) range:   {np.imag(z1).min():.3f} – {np.imag(z1).max():.3f}")
print(f"Im(z2) range:   {np.imag(z2).min():.3f} – {np.imag(z2).max():.3f}")

In [ ]:
import math
from jaxvacua.flux_bounding import bounded_fluxes

# Full Dataset B parameters (arXiv:2501.03984, Table 2)
# Used for bounding box analysis and the full scan.
_dil_B = (math.sqrt(3.) / 2., 10.)
sampler_B = jvc.data_sampler(
    model,
    moduli_bounds=(2., 5.),
    dilaton_bounds=_dil_B,
    axion_bounds=(-0.5, 0.5),
    seed=42,
)
bf_B = bounded_fluxes(model, sampler=sampler_B, Nmax=10)

print(f"n_fluxes     = {bf_B.n_fluxes}  (= 2*(h12+1))")
print(f"dimension_H3 = {bf_B.dimension_H3}  (= h12+1)")

### Bounding box for full Dataset B

We sample moduli points from the Dataset B region and compute the eigenvalue-based bounding box (§3.1 of arXiv:2501.03984).

| Bound | Formula | Quantity |
|---|---|---|
| `h1_box` | $\sqrt{N_{\max}/(s_{\min}\,\tilde\mu_{\min})}$ | max $\|h_1\|$ |
| `h2_box` | $\sqrt{N_{\max}/(s_{\min}\,\mu_{\min})}$ | max $\|h_2\|$ |
| `h_box`  | $\sqrt{2\lambda_{\max} N_{\max}/s_{\min}}$ | max $\|h\|$ (global) |

The full Cartesian-product enumeration $(2h_{1,\max}+1)^3\times(2h_{2,\max}+1)^3$ is infeasible for Dataset B due to the small $\tilde\mu_{\min}$ eigenvalue near $\mathrm{Im}(z)\sim 2$.

In [ ]:
# Pre-compute eigenvalue bounds once (reused by enumerate_fluxes)
h1_box_B, h2_box_B, h_box_B = bf_B.compute_eigenvalue_bounds(n_sample=1_000)

h1_max_B = int(np.ceil(h1_box_B))
h2_max_B = int(np.ceil(h2_box_B))
dim      = bf_B.dimension_H3
n_box_B  = (2*h1_max_B + 1)**dim * (2*h2_max_B + 1)**dim

print()
print(f"  h1_box = {h1_box_B:.2f}  → h1_max = {h1_max_B}")
print(f"  h2_box = {h2_box_B:.2f}  → h2_max = {h2_max_B}")
print(f"  h_box  = {h_box_B:.2f}")
print()
print(f"  Cartesian product: ({2*h1_max_B+1})³ × ({2*h2_max_B+1})³ ≈ {n_box_B:,}")
print(f"  → Streaming mode (h₂-outer) activates automatically in enumerate_fluxes")

### Test scan: N_max = 4, s ≥ 2

We restrict to **N_flux ≤ 4** and **Im(τ) ≥ 2** to get a small, feasible bounding box
while keeping a non-trivial ground truth.

From `dataset_B.p` (precomputed):
- N_flux ≤ 4, s ≥ 2, Im(z) ∈ [2,5] → **42 expected solutions** (exact subset of Dataset B).

These are the solutions our scan must find.  A 100 % match validates:
- the `linearised_shifts_H` pipeline for moduli-adaptive ISD completion,
- the Newton refinement convergence,
- and the flux conventions against the paper.

In [ ]:
import math
from jaxvacua.flux_bounding import bounded_fluxes

# -----------------------------------------------------------------------
# Test-scan parameters
# N_flux ≤ 4, Im(τ) ≥ 2 — 42 solutions expected from Dataset B
# -----------------------------------------------------------------------
Nmax_test        = 10
moduli_bounds_B  = (1.9, 5.1)               # same Im(z) region as Dataset B
dil_test         = (np.sqrt(3)/2, 10.)              # s_min=2 shrinks h-box relative to s_min=√3/2
axion_bounds     = (-0.51, 0.51)

sampler_test = jvc.data_sampler(
    model,
    moduli_bounds=moduli_bounds_B,
    dilaton_bounds=dil_test,
    axion_bounds=axion_bounds,
    seed=42,
)
bf_test = bounded_fluxes(model, sampler=sampler_test, Nmax=Nmax_test)

# Pre-compute eigenvalue bounds (reused by enumerate_fluxes — no re-sampling)
h1_box_t, h2_box_t, h_box_t = bf_test.compute_eigenvalue_bounds(n_sample=1_000)

h1_max_t = int(np.ceil(h1_box_t))
h2_max_t = int(np.ceil(h2_box_t))
dim      = bf_test.dimension_H3
n_box_t  = (2*h1_max_t + 1)**dim * (2*h2_max_t + 1)**dim

print()
print(f"  Nmax={Nmax_test},  s ∈ {dil_test},  Im(z) ∈ {moduli_bounds_B}")
print(f"  h1_box = {h1_box_t:.2f}  → h1_max = {h1_max_t}")
print(f"  h2_box = {h2_box_t:.2f}  → h2_max = {h2_max_t}")
print(f"  h_box  = {h_box_t:.2f}")
print(f"  Cartesian product: ({2*h1_max_t+1})³ × ({2*h2_max_t+1})³ ≈ {n_box_t:,}")

In [ ]:
@jit
def constraints_model(moduli,tau,flux):
    
    #return ((jnp.all(jnp.imag(moduli)<3.,axis=0))&(jnp.all(jnp.imag(moduli)>2.,axis=0)))
    return ((jnp.all(jnp.imag(moduli)>=1.,axis=0)))

In [ ]:
# Run enumerate_fluxes with ISD refinement via linearised_shifts_H.
#
# Key parameters:
#   use_linearised_shifts=True  — moves moduli to be self-consistent with each h
#                                 (Algorithm 1 of arXiv:2501.03984)
#   n_isd_iters=5               — 5 linearised_shifts_H iterations per starting point
#   moduli_regions              — Cartesian product across h12 moduli dimensions
#
# Eigenvalue bounds were pre-computed above → Step 1 is skipped.

enum_results_test = bf_test.enumerate_fluxes(
    n_sample=500,
    n_isd_per_h=30,
    max_h_candidates=5_000_000,
    refine=True,
    return_moduli=True,
    newton_tol=1e-10,
    newton_max_iters=10,
    newton_step_size=1.0,
    confirm_streaming=False,
    #moduli_regions=[(2., 3.), (3., 5.)],
    moduli_regions=[(2., 3.)],
    use_linearised_shifts=True,
    chunk_size=20_000,
    n_isd_iters=8,
    n_moduli_batches=30,
    constraints=constraints_model,
    verbose=True,
)
print(f"\nFound {len(enum_results_test)} converged SUSY vacua in test region.")

In [ ]:
# Display test scan results
if len(enum_results_test) == 0:
    print("No converged vacua found. Check that the model is FluxVacuaFinder")
    print("and that a_matrix is set correctly.")
else:
    print(f"{'#':>3}  {'Im(z1)':>8} {'Im(z2)':>8} {'Im(τ)':>8}  {'|W₀|':>12}  {'N_flux':>6}  flux [f|h]")
    print("-" * 100)
    for i, vac in enumerate(enum_results_test):
        fv = jnp.array(vac["flux"], dtype=float)
        mv = vac["moduli"]
        tv = vac["tau"]
        W0    = model.superpotential(mv, tv, fv)
        Nflux = abs(float(model.tadpole(fv).real))
        print(
            f"{i:>3}  "
            f"{float(jnp.imag(mv[0])):>8.4f} "
            f"{float(jnp.imag(mv[1])):>8.4f} "
            f"{float(jnp.imag(tv)):>8.4f}  "
            f"{abs(W0):>12.4e}  "
            f"{Nflux:>6.0f}  "
            f"{np.round(np.array(fv)).astype(int)}"
        )

### Direct verification: Newton refinement of stored Dataset B solutions

Newton-refine the first 500 stored solutions starting from the paper's VEVs.
All should converge with residual ≪ 10⁻¹⁰, confirming that:
- the `a_matrix` and prepotential match the paper,
- the symplectic conventions are consistent.

In [ ]:
n_verify = min(500, len(datsB))

moduli_0 = jnp.array(np.column_stack([z1[:n_verify], z2[:n_verify]]), dtype=complex)
tau_0    = jnp.array(tau_B[:n_verify], dtype=complex)
flux_0   = jnp.array(flux_B[:n_verify], dtype=float)

# bf_B is already built above (full Dataset B sampler, Nmax=10)
moduli_ref, tau_ref, residuals = bf_B.newton_refine_batch(
    moduli_0, tau_0, flux_0,
    step_size=1.0, tol=1e-12, max_iters=10,
)

res_np     = np.asarray(residuals)
n_conv     = int(np.sum(res_np < 1e-10))
dz1_drift  = np.abs(np.asarray(moduli_ref[:, 0]) - z1[:n_verify])
dz2_drift  = np.abs(np.asarray(moduli_ref[:, 1]) - z2[:n_verify])
dtau_drift = np.abs(np.asarray(tau_ref) - tau_B[:n_verify])

print(f"Verified {n_verify}/{len(datsB)} Dataset B solutions")
print(f"  Converged (res < 1e-10): {n_conv}/{n_verify}")
print(f"  Max residual:            {res_np.max():.2e}")
print(f"  Mean residual:           {res_np.mean():.2e}")
print()
print(f"  Max |Δz1| drift:  {dz1_drift.max():.2e}")
print(f"  Max |Δz2| drift:  {dz2_drift.max():.2e}")
print(f"  Max |Δτ|  drift:  {dtau_drift.max():.2e}")
print()
if n_conv == n_verify:
    print("✓  All solutions converged — model setup confirmed.")
else:
    print(f"⚠  {n_verify - n_conv} solutions did NOT converge.")
    print("   Check a_matrix and symplectic conventions.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Residual histogram
axes[0].hist(np.log10(res_np + 1e-16), bins=40, color="steelblue", edgecolor="white")
axes[0].axvline(-10, color="red", linestyle="--", label="tol = 1e-10")
axes[0].set_xlabel(r"$\log_{10}(\sum|D_I W|)$")
axes[0].set_ylabel("count")
axes[0].set_title(f"Newton residuals — {n_verify} Dataset B solutions")
axes[0].legend()

# Im(tau) distribution — full Dataset B
axes[1].hist(np.imag(tau_B), bins=50, color="coral", edgecolor="white")
axes[1].axvline(2.0, color="red", linestyle="--", label="test scan: s ≥ 2")
axes[1].set_xlabel(r"$s = \mathrm{Im}(\tau)$")
axes[1].set_ylabel("count")
axes[1].set_title("Full Dataset B — dilaton distribution")
axes[1].legend()
axes[1].set_yscale("log")

# N_flux distribution — full Dataset B
axes[2].hist(Nflux_B, bins=np.arange(0.5, 11.5, 1), color="mediumseagreen", edgecolor="white")
axes[2].axvline(4.5, color="red", linestyle="--", label="test scan: N_flux ≤ 4")
axes[2].set_xlabel(r"$N_\mathrm{flux}$")
axes[2].set_ylabel("count")
axes[2].set_title("Full Dataset B — tadpole distribution")
axes[2].legend()
axes[2].set_yscale("log")

plt.tight_layout()
plt.show()

# Count expected solutions in test region
mask_test = (Nflux_B <= Nmax_test) & (np.imag(tau_B) >= dil_test[0])
n_expected = int(mask_test.sum())
print(f"\nDataset B solutions in test region (N_flux≤{Nmax_test}, s≥{dil_test[0]:.0f}):")
print(f"  Expected: {n_expected}  |  Found: {len(enum_results_test)}")

### Cross-check: test scan results ⊆ Dataset B

For each flux vector found by `enumerate_fluxes`, check whether it (or its sign-flip)
appears in `dataset_B.p`.

**Note on equivalences handled:**
- **Sign flip**: `(f, h) → (−f, −h)` maps `W → −W`, leaving `|DW|=0` invariant.
  Both orientations represent the same physical vacuum.
- **Monodromy**: Re(z) shifts by integers transform the flux vector via the monodromy matrix.
  The paper reduces Re(z) ∈ [−½, ½], so our Newton-refined solutions should already be
  in canonical form; however small residual shifts may occur.

In [ ]:
from functools import partial
@jit
def monodromy_matrix_P11169(n1,n2):
    
    return jnp.array([[1,-n1,-n2,3*n2+n1/2*(3*n1**2+3*n1*n2+n2**2+17),(3*n1+n2)*(3*n1+n2+3)/2,3*n1/2*(n1+1)+n1*n2],
                      [0,1,0,-1/2*(3*n1+n2-3)*(3*n1+n2),-3*(3*n1+n2),-3*n1-n2],
                      [0,0,1,-1/2*n1*(3*n1+2*n2-3),-3*n1-n2,-n1],
                      [0,0,0,1,0,0],[0,0,0,n1,1,0],[0,0,0,n2,0,1]])

@partial(jit, static_argnums = (3,))
def apply_monodromy_transformation(self,moduli, fluxes,return_monodromy=False):
    
    re_moduli = np.real(moduli)
    re_moduli_int = -np.around(re_moduli,0)
    new_moduli = moduli+re_moduli_int
    
    M_mono = monodromy_matrix_P11169(*re_moduli_int)
    
    new_F = jnp.matmul(M_mono,fluxes[:self.n_fluxes])
    new_H = jnp.matmul(M_mono,fluxes[self.n_fluxes:])
    
    new_fluxes = jnp.append(new_F,new_H)
    
    
    if return_monodromy:
        return new_moduli, new_fluxes,M_mono
    else:
    
        return new_moduli, new_fluxes

In [ ]:
from monodromy import monodromy_matrix_single_LCS, monodromy_matrix_LCS

In [ ]:
M1 = monodromy_matrix_single_LCS(model.lcs_tree, 0) 
M2 = monodromy_matrix_single_LCS(model.lcs_tree, 1) 

In [ ]:
M1test = monodromy_matrix_P11169(1,0)
M2test = monodromy_matrix_P11169(0,1)
M1test-M1,M2test-M2

In [ ]:
M3 = monodromy_matrix_LCS(model.lcs_tree, jnp.array([1,1]))

In [ ]:
M3 = monodromy_matrix(model, jnp.array([1,1]))

In [ ]:
M3test = monodromy_matrix_P11169(1,1)
M3test-M3

In [ ]:
from tqdm.auto import tqdm
for n1 in tqdm(range(-10,10)):
    for n2 in range(-10,10):
        M3test = monodromy_matrix_P11169(n1,n2)
        M3 = monodromy_matrix_LCS(model.lcs_tree, jnp.array([n1,n2]))
        diff = jnp.max(jnp.abs(M3test-M3))
        if diff>0:
            print("Issue?")
            sys.exit()
            
        M3 = monodromy_matrix(model, jnp.array([n1,n2]))
        diff = jnp.max(jnp.abs(M3test-M3))
        if diff>0:
            print("Issue with general??")
            sys.exit()

In [ ]:
if len(enum_results_test) == 0:
    print("No test vacua — run the scan cell first.")
else:
    # Build O(1) lookup set from Dataset B flux vectors
    # Store both (f,h) and (-f,-h) so sign-flip matches succeed
    flux_B_int = np.round(flux_B).astype(int)   # (12196, 12)
    flux_B_set = {row.tobytes() for row in flux_B_int}
    flux_B_set.update({(-row).tobytes() for row in flux_B_int})

    n_match    = 0
    n_no_match = 0
    unmatched  = []
    updated_fluxes = []
    for vac in enum_results_test:
        flux_int = np.round(np.array(vac["flux"])).astype(int)
        z = vac["moduli"]
        
        ImZ = z.imag
        if jnp.any(ImZ<2) or jnp.any(ImZ>5):
            continue
        
        tau = vac["tau"]
        #flux_int = np.round(np.array(vac["flux"])).astype(int)
        DW = jnp.linalg.norm(model.DW(z,jnp.conj(z),tau,jnp.conj(tau),flux_int))
        #print(f"Old DW: ",DW)
        c0 = tau.real
        s = tau.imag
        if np.abs(c0)>0.5 or s<0.8 or np.abs(tau)<1:
            tau,flux_int = model.map_to_FD_tau(tau,flux_int)
            
        ReZ = z.real
        shift = -np.rint(ReZ)
        M = monodromy_matrix_P11169(*shift)
        new_F = np.matmul(M,flux_int[:model.n_fluxes])
        new_H = np.matmul(M,flux_int[model.n_fluxes:])
        new_fluxes = jnp.append(new_F,new_H)
        flux_int = np.array(new_fluxes).astype(int)
        z = z+shift
        DW = jnp.linalg.norm(model.DW(z,jnp.conj(z),tau,jnp.conj(tau),flux_int))
        if np.abs(DW)>1e-10:
            print("F-terms off?")
            sys.exit()
        ### Handle boundary points
        Rez1 = np.around(z.real[0],10)
        Rez2 = np.around(z.real[1],10)
        ReTau = np.around(tau.real,10)
        flag1 = (Rez1==-0.5)&(Rez2!=-0.5)
        flag2 = (Rez1!=-0.5)&(Rez2==-0.5)
        flag3 = (Rez1==-0.5)&(Rez2==-0.5)
        flag4 = ReTau==-0.5
        flags = [flag1,flag2,flag3,flag4]
        for j in range(len(flags)):
            flag = flags[j]
            if not flag:
                continue
            
            if j<=2:
                if j ==0:
                    z = z+np.array([1,0])

                    M_mono = np.array(monodromy_matrix_P11169(1,0))
                elif j==1:
                    z = z+np.array([0,1])

                    M_mono = np.array(monodromy_matrix_P11169(0,1))
                elif j==2:
                    z = z+np.array([1,1])

                    M_mono = np.array(monodromy_matrix_P11169(1,1))
                    
                new_F = np.matmul(M_mono,flux_int[:model.n_fluxes])
                new_H = np.matmul(M_mono,flux_int[model.n_fluxes:])
                new_fluxes = jnp.append(new_F,new_H)
                flux_int = np.array(new_fluxes).astype(int)
            else:
                new_H = flux_int[model.n_fluxes:]
                new_F = flux_int[:model.n_fluxes]+new_H
                new_fluxes = jnp.append(new_F,new_H)
                flux_int = np.array(new_fluxes).astype(int)
                tau = tau+1
                
            DW = jnp.linalg.norm(model.DW(z,jnp.conj(z),tau,jnp.conj(tau),flux_int))
            if np.abs(DW)>1e-10:
                print("F-terms off?")
                sys.exit()
                
                
        updated_fluxes.append(flux_int)
        if flux_int.tobytes() in flux_B_set:
            n_match += 1
        else:
            if (-flux_int).tobytes() in flux_B_set:
                print("OK")
            n_no_match += 1
            """
            ReZ = z.real
            ReT = tau.real
            if np.any(np.abs(np.around(ReZ,5))==0.5) or np.abs(np.around(ReT,5))==0.5:
                continue
            """
            unmatched.append([flux_int,z,tau])

    # Dataset B expected count in test region
    mask_test = (Nflux_B <= Nmax_test) & (np.imag(tau_B) >= dil_test[0])
    n_expected = int(mask_test.sum())
    
    updated_fluxes = np.unique(updated_fluxes,axis=0)

    print("=" * 55)
    print("Cross-check: test scan vs Dataset B")
    print("=" * 55)
    print(f"  Test scan found:          {len(enum_results_test)}")
    print(f"  Present in Dataset B:     {n_match}")
    print(f"  Not in Dataset B:         {n_no_match}")
    print(f"  Expected (from paper):    {n_expected}")
    print()

    if n_no_match == 0:
        print("✓  All found solutions are in Dataset B.")
    else:
        print(f"⚠  {n_no_match} flux vectors not found in Dataset B.")
        print("   Printing unmatched fluxes:")
        for fx in unmatched[:5]:
            print(f"     {fx}")
        if len(unmatched) > 5:
            print(f"     ... ({len(unmatched)-5} more)")
        print()
        print("   Possible causes:")
        print("   - Monodromy orbit not reduced (Re(z) not in [-½, ½]²)")
        print("   - Sign convention differs from paper")
        print("   - Genuine false positive (should not occur with Newton tol=1e-10)")

    print()
    if len(enum_results_test) == n_expected:
        print("✓  Complete: found exactly the expected number of solutions.")
    elif len(enum_results_test) < n_expected:
        missed = n_expected - len(enum_results_test)
        print(f"⚠  Incomplete: missed {missed}/{n_expected} solutions.")
        print("   Try increasing n_isd_iters or adding more moduli_regions.")
    else:
        print(f"⚠  Found more ({len(enum_results_test)}) than expected ({n_expected}).")
        print("   Check for duplicates or a wider moduli patch.")

### Pipeline validation: recover solutions from h-fluxes alone

Take a subset of Dataset B solutions with $N_{\rm flux} \leq 5$, extract **only their $h$-fluxes**, and verify that we can recover the full solution (f-fluxes, moduli, $\tau$) from scratch using randomly sampled starting points.

This validates the complete pipeline:
1. **ISD completion** (`isd_refine_batch`): given $h$ and *random* moduli/tau starting points, iteratively compute self-consistent $(z, \tau, f)$ via `linearised_shifts`.
2. **Newton refinement** (`newton_refine_batch`): refine the ISD candidate to solve $D_I W = 0$ exactly.
3. **Comparison**: verify the converged solution matches the stored Dataset B entry.

In [ ]:
# Select Dataset B solutions with N_flux <= 5, s >= 2
mask_pipe = (Nflux_B <= 5) & (np.imag(tau_B) >= 2.)
idx_pipe = np.where(mask_pipe)[0][:100]
n_pipe = len(idx_pipe)

h_pipe = jnp.array(h_B[idx_pipe], dtype=float)          # ONLY h-fluxes — no f, no moduli, no tau
f_pipe_ref = f_B[idx_pipe]                                # reference (for comparison only)
z_pipe_ref = np.column_stack([z1[idx_pipe], z2[idx_pipe]])
tau_pipe_ref = tau_B[idx_pipe]

print(f"Selected {n_pipe} h-flux vectors from Dataset B (N_flux ≤ 5, s ≥ 2)")
print(f"N_flux values: {sorted(set(Nflux_B[idx_pipe].astype(int)))}")
print(f"  → We use ONLY h_B — all moduli/tau starting points are random.")

In [ ]:
# Step 1: ISD completion with RANDOM starting points
# Multiple passes with different random moduli/tau to increase coverage,
# following the multi-pass strategy from run_hscan_34.py.

n_pts_per_pass = 200   # random starting points per pass
n_passes = 10         # number of passes with fresh random points
n_iters = 10          # linearised_shifts iterations per pass

# Build lookup set from reference f-fluxes (for evaluation only)
f_ref_int = np.round(f_pipe_ref).astype(int)

all_recovered = {}  # h_idx → best result

for p in range(n_passes):
    # Fresh random starting points each pass
    moduli_rand = jnp.array(sampler_B.get_complex_moduli(n_pts_per_pass), dtype=complex)
    tau_rand = jnp.array(sampler_B.get_complex_tau(n_pts_per_pass), dtype=complex)

    mod_out, tau_out, flux_out = bf_B.isd_refine_batch(
        h_pipe, moduli_rand, tau_rand, n_iters=n_iters,
    )

    # Check which h-vectors recovered the correct f-flux
    for i in range(n_pipe):
        if i in all_recovered:
            continue
        f_ref = f_ref_int[i]
        for j in range(mod_out.shape[1]):
            f_cand = np.round(flux_out[i, j, :6].real).astype(int)
            if np.array_equal(f_cand, f_ref) or np.array_equal(f_cand, -f_ref):
                all_recovered[i] = {
                    'idx': idx_pipe[i], 'pass': p,
                    'flux': np.round(flux_out[i, j].real).astype(float),
                    'moduli': mod_out[i, j],
                    'tau': tau_out[i, j],
                }
                break

    print(f"  Pass {p+1}/{n_passes}: {len(all_recovered)}/{n_pipe} h-vectors recovered "
          f"({n_pts_per_pass} random starts, {n_iters} iters)")

print(f"\nISD completion recovered correct f-flux: {len(all_recovered)}/{n_pipe} "
      f"using only random starting points")

In [ ]:
# Step 2: Newton-refine the recovered candidates and compare to Dataset B
recovered = list(all_recovered.values())

if recovered:
    flux_arr_pipe = jnp.array([r['flux'] for r in recovered])
    mod_arr_pipe = jnp.array([r['moduli'] for r in recovered], dtype=complex)
    tau_arr_pipe = jnp.array([r['tau'] for r in recovered], dtype=complex)

    mod_newton, tau_newton, res_newton = bf_B.newton_refine_batch(
        mod_arr_pipe, tau_arr_pipe, flux_arr_pipe,
        step_size=1.0, tol=1e-12, max_iters=200,
    )

    res_np = np.asarray(res_newton)
    n_conv = int(np.sum(res_np < 1e-10))

    # Compare to Dataset B reference
    print(f"Newton converged: {n_conv}/{len(recovered)}")
    print(f"Residuals: max={res_np.max():.2e}, min={res_np.min():.2e}\n")

    print(f"{'#':>3} {'pass':>4} {'N_flux':>6} {'|ΣD_IW|':>10} {'|Δz|_max':>10} {'|Δτ|':>10}  match?")
    print("-" * 66)
    n_exact = 0
    for k, r in enumerate(recovered):
        hi = list(all_recovered.keys())[k]
        z_ref_k = z_pipe_ref[hi]
        t_ref_k = tau_pipe_ref[hi]
        Nfl = float(Nflux_B[r['idx']])

        dz = float(np.max(np.abs(np.asarray(mod_newton[k]) - z_ref_k)))
        dt = float(np.abs(np.asarray(tau_newton[k]) - t_ref_k))
        close = dz < 0.01 and dt < 0.01 and res_np[k] < 1e-10
        if close:
            n_exact += 1
        print(f"{k:>3} {r['pass']:>4} {Nfl:>6.0f} {res_np[k]:>10.2e} {dz:>10.6f} {dt:>10.6f}  "
              f"{'YES' if close else 'no'}")

    print(f"\n{'='*66}")
    print(f"Pipeline validation: {n_exact}/{n_pipe} Dataset B solutions recovered")
    print(f"  from h-fluxes alone with random starting points")
    print(f"  ({n_passes} passes × {n_pts_per_pass} random moduli/tau per pass)")
    if n_exact == len(recovered):
        print(f"  All {n_exact} recovered solutions match Dataset B exactly.")
else:
    print("No solutions recovered — try increasing n_passes or n_pts_per_pass.")

### Full Dataset B scan

**Parameters:** Im(zⁱ) ∈ [2,5], s ∈ [√3/2, 10], N_max=10, `use_linearised_shifts=True`

| Step | Status | Notes |
|---|---|---|
| Model + `a_matrix` | ✅ | Verified via direct Newton refinement |
| Bounding box | ✅ | `compute_bounding_box` |
| h₂-outer streaming | ✅ | `_iter_h_chunks_streaming` + `_stream_fixed_chunks` |
| `linearised_shifts_H` ISD completion | ✅ | `isd_refine_batch`, `use_linearised_shifts=True` |
| Newton refinement | ✅ | `newton_refine_batch` |
| Monodromy reduction | ❌ | Low priority — small effect (~1% reduction) |

**Expected:** ~12,196 SUSY vacua; ~4.64 billion h-flux candidates before filtering.
The streaming mode keeps peak memory at ~2.4 MB (one 100K-row h-chunk at a time).

⚠ **Runtime:** hours to days on CPU; ~1 hour on a GPU cluster with h₂-batch sharding.
Set `confirm_streaming=False` for non-interactive execution.

In [ ]:
# Full Dataset B scan — set RUN_FULL_SCAN = True to execute
# Expected runtime: several hours (CPU) or ~1 h (GPU cluster with h2-sharding)
RUN_FULL_SCAN = False

if RUN_FULL_SCAN:
    import math as _math
    from jaxvacua.flux_bounding import bounded_fluxes as _bf

    sampler_full = jvc.data_sampler(
        model,
        moduli_bounds=(2., 5.),
        dilaton_bounds=(_math.sqrt(3.) / 2., 10.),
        axion_bounds=(-0.5, 0.5),
        seed=42,
    )
    bf_full = _bf(model, sampler=sampler_full, Nmax=10)

    full_results = bf_full.enumerate_fluxes(
        n_sample=1000,
        n_isd_per_h=10,
        max_h_candidates=10_000_000,   # streaming triggered immediately for Dataset B
        refine=True,
        return_moduli=True,
        newton_tol=1e-10,
        newton_max_iters=100,
        newton_step_size=1.0,
        confirm_streaming=False,
        moduli_regions=[(2., 3.), (3., 4.), (4., 5.)],
        use_linearised_shifts=True,
        n_isd_iters=5,
        verbose=True,
    )
    print(f"\nFull scan: found {len(full_results)} vacua  (expected: 12,196)")
else:
    print("RUN_FULL_SCAN = False — skipping full Dataset B scan.")
    print("Set RUN_FULL_SCAN = True to execute (long runtime).")

In [ ]:
# Visual comparison: VEV distributions of test-scan results vs Dataset B subset
import matplotlib.pyplot as plt

mask_test = (Nflux_B <= Nmax_test) & (np.imag(tau_B) >= dil_test[0])
z1_sub  = z1[mask_test]
z2_sub  = z2[mask_test]
tau_sub = tau_B[mask_test]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Im(z1) comparison
axes[0].hist(np.imag(z1_sub),  bins=20, alpha=0.6, label="Dataset B subset", color="steelblue", edgecolor="white")
if enum_results_test:
    imz1_found = [float(jnp.imag(v["moduli"][0])) for v in enum_results_test]
    axes[0].hist(imz1_found, bins=20, alpha=0.7, label="enumerate_fluxes", color="coral", edgecolor="white")
axes[0].set_xlabel(r"$\mathrm{Im}(z_1)$"); axes[0].set_ylabel("count")
axes[0].set_title(r"$\mathrm{Im}(z_1)$ distribution"); axes[0].legend()

# Im(z2) comparison
axes[1].hist(np.imag(z2_sub),  bins=20, alpha=0.6, label="Dataset B subset", color="steelblue", edgecolor="white")
if enum_results_test:
    imz2_found = [float(jnp.imag(v["moduli"][1])) for v in enum_results_test]
    axes[1].hist(imz2_found, bins=20, alpha=0.7, label="enumerate_fluxes", color="coral", edgecolor="white")
axes[1].set_xlabel(r"$\mathrm{Im}(z_2)$"); axes[1].set_ylabel("count")
axes[1].set_title(r"$\mathrm{Im}(z_2)$ distribution"); axes[1].legend()

# Im(tau) comparison
axes[2].hist(np.imag(tau_sub), bins=20, alpha=0.6, label="Dataset B subset", color="steelblue", edgecolor="white")
if enum_results_test:
    imtau_found = [float(jnp.imag(v["tau"])) for v in enum_results_test]
    axes[2].hist(imtau_found, bins=20, alpha=0.7, label="enumerate_fluxes", color="coral", edgecolor="white")
axes[2].set_xlabel(r"$s = \mathrm{Im}(\tau)$"); axes[2].set_ylabel("count")
axes[2].set_title(r"$s$ distribution"); axes[2].legend()

plt.suptitle(f"Test region: N_flux ≤ {Nmax_test}, s ≥ {dil_test[0]:.0f}, Im(z) ∈ {moduli_bounds_B}", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Final summary table
print("=" * 60)
print("RECOVERY STATUS — Dataset B (arXiv:2501.03984)")
print("=" * 60)
print()
print(f"  Full Dataset B:         {len(datsB):>6} solutions")
print()
print(f"  Test region summary")
print(f"  ├─ Parameters:          N_flux ≤ {Nmax_test}, s ≥ {dil_test[0]:.0f}, Im(z) ∈ {moduli_bounds_B}")
print(f"  ├─ Expected (paper):    {int(mask_test.sum()):>6}")
print(f"  ├─ Found (scan):        {len(enum_results_test):>6}")

if len(enum_results_test) > 0:
    n_in_B = sum(
        1 for v in enum_results_test
        if np.round(np.array(v["flux"])).astype(int).tobytes() in flux_B_set
    )
    print(f"  ├─ Matched in Dataset B:{n_in_B:>6}")
    recall = len(enum_results_test) / int(mask_test.sum()) if mask_test.sum() > 0 else 0
    precision = n_in_B / len(enum_results_test) if enum_results_test else 0
    print(f"  ├─ Recall:              {recall*100:.1f}%")
    print(f"  └─ Precision:           {precision*100:.1f}%")
else:
    print(f"  └─ (scan not yet run)")

print()
print("  Full scan status:       ", end="")
if RUN_FULL_SCAN and 'full_results' in dir():
    n_full_in_B = sum(
        1 for v in full_results
        if np.round(np.array(v["flux"])).astype(int).tobytes() in flux_B_set
    )
    print(f"{len(full_results)} found, {n_full_in_B} matched in Dataset B")
else:
    print("NOT RUN (set RUN_FULL_SCAN=True)")